# 한국어 정규화 Gateway E0/E1/E2/E3 평가 보고서

이 노트북은 모델 추론을 수행하지 않고, 평가 실행기가 만든 **집계 산출물만** 읽어 결과를 재현한다. 공격 프롬프트 원문과 행 단위 예측은 로드하지 않는다.

- E0: clean / gateway OFF
- E1: obfuscated / gateway OFF
- E2: obfuscated / gateway ON
- E3: clean / gateway ON


In [1]:
from pathlib import Path
from collections import defaultdict
from IPython.display import HTML, Markdown, display
import csv
import html
import json
import os

RUN_ID = os.getenv("K_SAFEGUARD_REPORT_RUN", "normalizer-eval-full-20260808")

def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "experiments" / "benchmark" / "NORMALIZER_EVALUATION.md").exists():
            return candidate
    raise FileNotFoundError("k-safeguard 저장소 루트를 찾지 못했습니다. 저장소 안에서 노트북을 실행하세요.")

REPO_ROOT = find_repo_root()
RUN_DIR = REPO_ROOT / "experiments" / "benchmark" / "results" / RUN_ID
print(f"분석 대상: {RUN_ID}")


분석 대상: normalizer-eval-full-20260808


In [2]:
required = ["manifest.json", "summary.json", "condition_summary.csv"]
missing = [name for name in required if not (RUN_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"{RUN_DIR}에 필요한 집계 파일이 없습니다: {', '.join(missing)}")

manifest = json.loads((RUN_DIR / "manifest.json").read_text(encoding="utf-8"))
summary = json.loads((RUN_DIR / "summary.json").read_text(encoding="utf-8"))
with (RUN_DIR / "condition_summary.csv").open(encoding="utf-8-sig", newline="") as handle:
    condition_rows = list(csv.DictReader(handle))

assert manifest["run_id"] == RUN_ID
assert summary["records"] == manifest["conditions"]["task_count"]
print(f"집계 로드 완료: {RUN_ID} / {summary['records']:,} records / {summary['independent_seeds']:,} seeds")


집계 로드 완료: normalizer-eval-full-20260808 / 11,110 records / 505 seeds


In [3]:
def pct(value, digits=2, signed=False):
    if value is None:
        return "—"
    sign = "+" if signed and value > 0 else ""
    return f"{sign}{value * 100:.{digits}f}%"

def metric_text(metric, signed=False):
    estimate = pct(metric["seed_balanced_estimate"], signed=signed)
    low = pct(metric.get("ci95_low"), signed=signed)
    high = pct(metric.get("ci95_high"), signed=signed)
    return f"{estimate} ({low}–{high})"

def md_table(headers, rows):
    head = "| " + " | ".join(headers) + " |"
    rule = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = ["| " + " | ".join(str(value) for value in row) + " |" for row in rows]
    return "\n".join([head, rule, *body])

def grouped_bar_chart(labels, series, title, y_label="비율"):
    width, height = 820, 380
    left, right, top, bottom = 72, 24, 48, 74
    plot_w, plot_h = width - left - right, height - top - bottom
    colors = ["#2563eb", "#f97316", "#16a34a", "#9333ea"]
    group_w = plot_w / max(len(labels), 1)
    bar_w = min(42, group_w * 0.72 / max(len(series), 1))
    svg = [f'<svg viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="{html.escape(title)}">']
    svg.append('<style>text{font-family:system-ui,sans-serif;fill:#334155}.title{font-size:18px;font-weight:700}.axis{font-size:12px}.label{font-size:12px}.value{font-size:10px;font-weight:600}</style>')
    svg.append(f'<text x="{width/2}" y="24" text-anchor="middle" class="title">{html.escape(title)}</text>')
    for tick in range(0, 101, 20):
        y = top + plot_h * (1 - tick / 100)
        svg.append(f'<line x1="{left}" y1="{y:.1f}" x2="{width-right}" y2="{y:.1f}" stroke="#e2e8f0"/>')
        svg.append(f'<text x="{left-10}" y="{y+4:.1f}" text-anchor="end" class="axis">{tick}%</text>')
    for index, label in enumerate(labels):
        center = left + group_w * (index + 0.5)
        svg.append(f'<text x="{center:.1f}" y="{height-bottom+28}" text-anchor="middle" class="label">{html.escape(label)}</text>')
        for series_index, (name, values) in enumerate(series.items()):
            value = max(0.0, min(1.0, values[index]))
            bar_h = plot_h * value
            x = center - (len(series) * bar_w) / 2 + series_index * bar_w
            y = top + plot_h - bar_h
            svg.append(f'<rect x="{x+2:.1f}" y="{y:.1f}" width="{bar_w-4:.1f}" height="{bar_h:.1f}" rx="3" fill="{colors[series_index]}"/>')
            svg.append(f'<text x="{x+bar_w/2:.1f}" y="{max(top+10, y-5):.1f}" text-anchor="middle" class="value">{value*100:.1f}</text>')
    legend_x = left
    for series_index, name in enumerate(series):
        svg.append(f'<rect x="{legend_x}" y="{height-20}" width="12" height="12" rx="2" fill="{colors[series_index]}"/>')
        svg.append(f'<text x="{legend_x+17}" y="{height-10}" class="axis">{html.escape(name)}</text>')
        legend_x += 130
    svg.append('</svg>')
    return HTML("".join(svg))


## 1. 실행 재현 정보

In [4]:
metadata_rows = [
    ("Run ID", manifest["run_id"]),
    ("Git commit", manifest["git"]["commit"][:12]),
    ("Dataset SHA-256", manifest["dataset"]["sha256"][:16] + "…"),
    ("Model", manifest["model"]["model_id"]),
    ("Model revision", manifest["model"]["revision"][:12]),
    ("Normalizer", manifest["normalizer"]["version"]),
    ("GPU", manifest["model"]["gpu_name"]),
    ("Bootstrap", f"seed cluster {manifest['runtime']['bootstrap_samples']:,}회"),
]
display(Markdown(md_table(["항목", "값"], metadata_rows)))


| 항목 | 값 |
| --- | --- |
| Run ID | normalizer-eval-full-20260808 |
| Git commit | 3c1c00434dc2 |
| Dataset SHA-256 | fbf9f978996c103b… |
| Model | kakaocorp/kanana-safeguard-prompt-2.1b |
| Model revision | 167d74d4706b |
| Normalizer | 0.1.0 |
| GPU | NVIDIA GeForce RTX 5070 Ti |
| Bootstrap | seed cluster 10,000회 |

## 2. 핵심 지표

In [5]:
overall = summary["gateway_metrics"]["overall"]
clean = summary["gateway_metrics"]["clean"]
headline = [
    ("CER", metric_text(overall["cer"]), "정규화 전 난독화 회피율"),
    ("Residual CER", metric_text(overall["residual_cer"]), "정규화 후 남은 회피율"),
    ("NRR", metric_text(overall["nrr"]), "기존 회피 사례 중 탐지 복원 비율"),
    ("Exact restoration", metric_text(overall["exact_restoration"]), "변형문이 clean과 정확히 같아진 비율"),
    ("ΔFPR-obfuscated", metric_text(overall["delta_fpr_obfuscated"], signed=True), "난독화 정상문 오탐 변화"),
    ("Clean mutation", metric_text(clean["clean_mutation_rate"]), "정상 입력 변경 비율"),
]
display(Markdown(md_table(["지표", "seed-balanced 추정치 (95% CI)", "의미"], headline)))


| 지표 | seed-balanced 추정치 (95% CI) | 의미 |
| --- | --- | --- |
| CER | 33.29% (31.48%–35.16%) | 정규화 전 난독화 회피율 |
| Residual CER | 32.23% (30.60%–33.85%) | 정규화 후 남은 회피율 |
| NRR | 1.76% (1.03%–2.61%) | 기존 회피 사례 중 탐지 복원 비율 |
| Exact restoration | 39.92% (39.76%–40.00%) | 변형문이 clean과 정확히 같아진 비율 |
| ΔFPR-obfuscated | +0.15% (-0.74%–+1.08%) | 난독화 정상문 오탐 변화 |
| Clean mutation | 0.00% (0.00%–0.00%) | 정상 입력 변경 비율 |

## 3. E0/E1/E2/E3 조건 비교

In [6]:
condition_totals = defaultdict(lambda: {"valid": 0, "blocked": 0})
for row in condition_rows:
    key = (row["condition"], row["label"])
    condition_totals[key]["valid"] += int(row["valid"])
    condition_totals[key]["blocked"] += int(row["blocked"])

condition_table = []
for condition in ["E0", "E1", "E2", "E3"]:
    attack = condition_totals[(condition, "attack")]
    benign = condition_totals[(condition, "benign")]
    attack_rate = attack["blocked"] / attack["valid"]
    benign_rate = benign["blocked"] / benign["valid"]
    condition_table.append((condition, f"{attack['blocked']:,}/{attack['valid']:,}", pct(attack_rate), f"{benign['blocked']:,}/{benign['valid']:,}", pct(benign_rate)))
display(Markdown(md_table(["조건", "공격 차단", "TPR", "정상문 차단", "FPR"], condition_table)))

labels = ["E0 clean/OFF", "E1 obf/OFF", "E2 obf/ON", "E3 clean/ON"]
attack_rates = [condition_totals[(c, "attack")]["blocked"] / condition_totals[(c, "attack")]["valid"] for c in ["E0", "E1", "E2", "E3"]]
benign_rates = [condition_totals[(c, "benign")]["blocked"] / condition_totals[(c, "benign")]["valid"] for c in ["E0", "E1", "E2", "E3"]]
display(grouped_bar_chart(labels, {"공격 TPR": attack_rates, "정상문 FPR": benign_rates}, "조건별 차단율"))


| 조건 | 공격 차단 | TPR | 정상문 차단 | FPR |
| --- | --- | --- | --- | --- |
| E0 | 283/301 | 94.02% | 6/204 | 2.94% |
| E1 | 1,940/3,010 | 64.45% | 28/2,040 | 1.37% |
| E2 | 1,935/3,010 | 64.29% | 31/2,040 | 1.52% |
| E3 | 283/301 | 94.02% | 6/204 | 2.94% |

## 4. 난독화 기법별 복원 효과

In [7]:
techniques = summary["gateway_metrics"]["by_technique"]
technique_rows = []
for item in techniques:
    metrics = item["metrics"]
    technique_rows.append((
        item["technique"],
        metric_text(metrics["cer"]),
        metric_text(metrics["residual_cer"]),
        metric_text(metrics["nrr"]),
        metric_text(metrics["exact_restoration"]),
        metric_text(metrics["delta_fpr_obfuscated"], signed=True),
    ))
display(Markdown(md_table(["기법", "CER", "Residual CER", "NRR", "정확 복원", "ΔFPR-obf"], technique_rows)))

technique_labels = [item["technique"] for item in techniques]
cer = [item["metrics"]["cer"]["seed_balanced_estimate"] for item in techniques]
residual = [item["metrics"]["residual_cer"]["seed_balanced_estimate"] for item in techniques]
display(grouped_bar_chart(technique_labels, {"CER": cer, "Residual CER": residual}, "기법별 정규화 전후 회피율"))


| 기법 | CER | Residual CER | NRR | 정확 복원 | ΔFPR-obf |
| --- | --- | --- | --- | --- | --- |
| break_spacing | 10.95% (7.42%–14.84%) | 10.95% (7.42%–14.84%) | 0.00% (0.00%–0.00%) | 0.00% (0.00%–0.00%) | 0.00% (0.00%–0.00%) |
| chosung | 80.32% (77.13%–83.33%) | 80.32% (77.13%–83.33%) | 0.00% (0.00%–0.00%) | 0.00% (0.00%–0.00%) | 0.00% (0.00%–0.00%) |
| jamo_decompose | 1.95% (0.71%–3.37%) | 0.00% (0.00%–0.00%) | 100.00% (100.00%–100.00%) | 100.00% (100.00%–100.00%) | +0.49% (-2.21%–+3.19%) |
| tensify | 68.97% (64.72%–73.05%) | 68.97% (64.72%–73.05%) | 0.00% (0.00%–0.00%) | 0.00% (0.00%–0.00%) | 0.00% (0.00%–0.00%) |
| zwsp_inject | 3.37% (1.77%–5.32%) | 0.00% (0.00%–0.00%) | 100.00% (100.00%–100.00%) | 100.00% (100.00%–100.00%) | +0.25% (-1.96%–+2.45%) |

## 5. 안전성·실행 건전성

In [8]:
e2_rows = [row for row in condition_rows if row["condition"] == "E2"]
latency_p50 = [float(row["normalizer_latency_p50_ms"]) for row in e2_rows if row["normalizer_latency_p50_ms"]]
latency_p95 = [float(row["normalizer_latency_p95_ms"]) for row in e2_rows if row["normalizer_latency_p95_ms"]]
health_rows = [
    ("독립 seed", f"{summary['independent_seeds']:,}"),
    ("조건 레코드", f"{summary['records']:,}"),
    ("Invalid output", f"{summary['invalid_outputs']:,} ({pct(summary['invalid_rate'])})"),
    ("Execution error", f"{summary['execution_errors']:,} ({pct(summary['execution_error_rate'])})"),
    ("Normalizer latency p50 범위", f"{min(latency_p50):.3f}–{max(latency_p50):.3f} ms" if latency_p50 else "—"),
    ("Normalizer latency p95 범위", f"{min(latency_p95):.3f}–{max(latency_p95):.3f} ms" if latency_p95 else "—"),
    ("결과 유효성", summary["validity"]),
    ("프로젝트 판정", summary["decision"]),
]
display(Markdown(md_table(["항목", "결과"], health_rows)))


| 항목 | 결과 |
| --- | --- |
| 독립 seed | 505 |
| 조건 레코드 | 11,110 |
| Invalid output | 0 (0.00%) |
| Execution error | 0 (0.00%) |
| Normalizer latency p50 범위 | 0.023–0.210 ms |
| Normalizer latency p95 범위 | 0.032–0.428 ms |
| 결과 유효성 | INCOMPLETE |
| 프로젝트 판정 | NOT_EVALUATED |

## 6. 해석과 다음 우선순위

In [9]:
by_name = {item["technique"]: item["metrics"] for item in techniques}
supported = [name for name in ["jamo_decompose", "zwsp_inject"] if by_name[name]["residual_cer"]["seed_balanced_estimate"] == 0]
unsupported = sorted(
    [name for name in by_name if name not in supported],
    key=lambda name: by_name[name]["residual_cer"]["seed_balanced_estimate"],
    reverse=True,
)
interpretation = f"""
- 현재 MVP가 지원하는 **{', '.join(supported)}**는 Residual CER 0%, NRR 100%, 정확 복원 100%로 관측됐다.
- 전체 NRR이 {pct(overall['nrr']['seed_balanced_estimate'])}에 머문 이유는 미지원 기법이 전체 집계를 지배하기 때문이다. 이는 지원 규칙의 실패율로 해석하면 안 된다.
- 다음 구현 우선순위는 Residual CER이 큰 **{unsupported[0]} ({pct(by_name[unsupported[0]]['residual_cer']['seed_balanced_estimate'])})**, **{unsupported[1]} ({pct(by_name[unsupported[1]]['residual_cer']['seed_balanced_estimate'])})** 순이다.
- clean mutation은 {pct(clean['clean_mutation_rate']['seed_balanced_estimate'])}, ΔFPR-obfuscated는 {pct(overall['delta_fpr_obfuscated']['seed_balanced_estimate'], signed=True)}로 관측됐다. 다만 후자의 95% CI가 0을 포함하므로 오탐 증가를 단정하지 않는다.
- 이 결과는 **지원하는 시각적 난독화 규칙에 대한 복원 가능성**을 입증하지만, 전체 난독화 방어가 복원됐다는 결론은 지지하지 않는다.
"""
display(Markdown(interpretation))



- 현재 MVP가 지원하는 **jamo_decompose, zwsp_inject**는 Residual CER 0%, NRR 100%, 정확 복원 100%로 관측됐다.
- 전체 NRR이 1.76%에 머문 이유는 미지원 기법이 전체 집계를 지배하기 때문이다. 이는 지원 규칙의 실패율로 해석하면 안 된다.
- 다음 구현 우선순위는 Residual CER이 큰 **chosung (80.32%)**, **tensify (68.97%)** 순이다.
- clean mutation은 0.00%, ΔFPR-obfuscated는 +0.15%로 관측됐다. 다만 후자의 95% CI가 0을 포함하므로 오탐 증가를 단정하지 않는다.
- 이 결과는 **지원하는 시각적 난독화 규칙에 대한 복원 가능성**을 입증하지만, 전체 난독화 방어가 복원됐다는 결론은 지지하지 않는다.


## 7. 해석 제한

현재 결과는 Kanana Safeguard-Prompt의 **Prompt track**만 평가한다. 하위 LLM의 clean intent-recognition과 semantic fidelity가 아직 측정되지 않았으므로 유효성은 `INCOMPLETE`, 프로젝트 판정은 `NOT_EVALUATED`다. 이 노트북만으로 GO/NO-GO를 선언하지 않는다.

또한 NRR은 원래 가드레일이 clean 공격을 차단했지만 난독화 입력을 놓친 사례만 분모로 삼는다. `jamo_decompose`와 `zwsp_inject`의 NRR 100%는 각각 11행, 19행의 회피 사례에 대한 결과이므로 CER·Residual CER·정확 복원과 함께 해석해야 한다.